# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Abdul-Rafay-246/internship-flyrank/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

1. **Source row:** One row is one pseudonymized content page for one client on one day.
2. **Final row:** After aggregation, one row is one pseudonymized content page.
3. **Table and window:** I use `fact_content_daily_performance` for March 2026. March 1-21 is the feature window, and March 22-31 is the future target window.
4. **Target and output:** I estimate whether a page's average daily impressions will decline by more than 20%. The output is a ranked review queue for a content reviewer.
5. **Deliberate exclusion:** Target-window measurements are not model features because they happen after the decision moment.

In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
%pip -q install duckdb huggingface_hub scikit-learn

import os
import duckdb
import pandas as pd

HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        HF_TOKEN = None

if not HF_TOKEN:
    raise ValueError("Add HF_TOKEN to Colab Secrets before running.")

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

release = "hf://datasets/FlyRank/internship-warehouse"

march_data = (
    f"read_parquet('{release}/"
    f"fact_content_daily_performance/month=2026-03/*.parquet')"
)

print("Connected to the March 2026 warehouse partition.")

Note: you may need to restart the kernel to use updated packages.
Connected to the March 2026 warehouse partition.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

The five model features use only information available by March 21. The future label is measured from March 22-31. Hash IDs are context for grouping and splitting, not model features. The future ratio is excluded after the leakage demonstration because it uses information from the target window.

In [22]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

field_contract = pd.DataFrame([
    ["feature_avg_daily_impressions", "Feature",
     "Knowable because it uses impressions from March 1-21 only."],
    ["feature_clicks", "Feature",
     "Knowable because it uses clicks recorded by March 21."],
    ["feature_ctr", "Feature",
     "Knowable because it uses earlier clicks and impressions."],
    ["feature_avg_position", "Feature",
     "Knowable because it uses positions recorded by March 21."],
    ["feature_active_days", "Feature",
     "Knowable because it counts active days before March 22."],
    ["future_decline_label", "Label",
     "Measured from March 22-31 after the decision moment."],
    ["future_avg_daily_impressions", "Label",
     "Used only to calculate the future-decline label."],
    ["client_hash_id", "Context",
     "Used only to keep clients separated during testing."],
    ["content_hash_id", "Context",
     "Used only to identify each pseudonymized page."],
    ["leaky_future_ratio", "Excluded",
     "Removed because it uses future information and reveals the label."]
], columns=["field", "type", "reason"])

field_contract

,field,type,reason
0,feature_avg_daily_impressions,Feature,Knowable because it uses impressions from Marc...
1,feature_clicks,Feature,Knowable because it uses clicks recorded by Ma...
2,feature_ctr,Feature,Knowable because it uses earlier clicks and im...
3,feature_avg_position,Feature,Knowable because it uses positions recorded by...
4,feature_active_days,Feature,Knowable because it counts active days before ...
5,future_decline_label,Label,Measured from March 22-31 after the decision m...
6,future_avg_daily_impressions,Label,Used only to calculate the future-decline label.
7,client_hash_id,Context,Used only to keep clients separated during tes...
8,content_hash_id,Context,Used only to identify each pseudonymized page.
9,leaky_future_ratio,Excluded,Removed because it uses future information and...


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [23]:
# Verification query 1 of 3: check for duplicate page-day rows.
grain_check = con.sql(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS row_count
    FROM {march_data}
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

print("Duplicate page-day groups:", len(grain_check))
grain_check

Duplicate page-day groups: 0


,report_date,client_hash_id,content_hash_id,row_count


In [24]:
# Verification query 2 of 3: count the slice and check dates.
slice_summary = con.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        COUNT(DISTINCT content_hash_id) AS content_pages,
        COUNT(DISTINCT client_hash_id) AS clients,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date
    FROM {march_data}
""").df()

slice_summary

,row_count,content_pages,clients,first_date,last_date
0,9841378,331437,55,2026-03-01,2026-03-31


In [25]:
# Verification query 3 of 3: check data availability.
availability = con.sql(f"""
    SELECT
        COUNT(*) AS all_rows,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS gsc_available_rows,
        COUNT(*) FILTER (
            WHERE ga4_data_available IS TRUE
        ) AS ga4_available_rows,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
              AND ga4_data_available IS TRUE
        ) AS both_available_rows
    FROM {march_data}
""").df()

availability

,all_rows,gsc_available_rows,ga4_available_rows,both_available_rows
0,9841378,3611061,413966,364347


In [26]:
# Combine daily rows into one row per page and create five features.
feature_frame = con.sql(f"""
    WITH page_windows AS (
        SELECT
            client_hash_id,
            content_hash_id,

            SUM(CASE
                WHEN report_date BETWEEN DATE '2026-03-01'
                                     AND DATE '2026-03-21'
                THEN gsc_impressions ELSE 0
            END) AS past_impressions,

            SUM(CASE
                WHEN report_date BETWEEN DATE '2026-03-01'
                                     AND DATE '2026-03-21'
                THEN gsc_clicks ELSE 0
            END) AS past_clicks,

            AVG(CASE
                WHEN report_date BETWEEN DATE '2026-03-01'
                                     AND DATE '2026-03-21'
                 AND gsc_impressions > 0
                THEN gsc_avg_position
            END) AS past_position,

            COUNT(DISTINCT CASE
                WHEN report_date BETWEEN DATE '2026-03-01'
                                     AND DATE '2026-03-21'
                 AND gsc_impressions > 0
                THEN report_date
            END) AS active_days,

            SUM(CASE
                WHEN report_date BETWEEN DATE '2026-03-22'
                                     AND DATE '2026-03-31'
                THEN gsc_impressions ELSE 0
            END) AS future_impressions,

            COUNT(DISTINCT CASE
                WHEN report_date BETWEEN DATE '2026-03-01'
                                     AND DATE '2026-03-21'
                THEN report_date
            END) AS past_days,

            COUNT(DISTINCT CASE
                WHEN report_date BETWEEN DATE '2026-03-22'
                                     AND DATE '2026-03-31'
                THEN report_date
            END) AS future_days

        FROM {march_data}
        WHERE gsc_data_available IS TRUE
        GROUP BY client_hash_id, content_hash_id
    )

    SELECT
        client_hash_id,
        content_hash_id,

        past_impressions / 21.0
            AS feature_avg_daily_impressions,

        past_clicks AS feature_clicks,

        100.0 * past_clicks / NULLIF(past_impressions, 0)
            AS feature_ctr,

        past_position AS feature_avg_position,
        active_days AS feature_active_days,

        future_impressions / 10.0
            AS future_avg_daily_impressions,

        CASE
            WHEN future_impressions / 10.0
                 < 0.8 * (past_impressions / 21.0)
            THEN 1
            ELSE 0
        END AS future_decline_label

    FROM page_windows
    WHERE past_days = 21
      AND future_days = 10
      AND past_impressions >= 100
""").df()

print("Feature-frame rows:", len(feature_frame))
display(feature_frame.head())

# Confirm that each client-page pair appears only once.
duplicate_feature_rows = feature_frame.duplicated(
    subset=["client_hash_id", "content_hash_id"]
).sum()

print("Duplicate rows in feature frame:", duplicate_feature_rows)

Feature-frame rows: 60796


,client_hash_id,content_hash_id,feature_avg_daily_impressions,feature_clicks,feature_ctr,feature_avg_position,feature_active_days,future_avg_daily_impressions,future_decline_label
0,client_3197e6291363b4db,content_c711a43ee1459d9e,39.714286,1.0,0.119904,8.342208,21,32.4,0
1,client_3197e6291363b4db,content_8e6f57489e905fed,8.571429,1.0,0.555556,4.695042,21,9.9,0
2,client_3197e6291363b4db,content_6f581bd395cd6e77,5.714286,0.0,0.000000,13.337982,21,5.7,0
3,client_3197e6291363b4db,content_ff28e181072e8eb6,89.904762,3.0,0.158898,7.679143,21,38.7,1
4,client_3197e6291363b4db,content_542589b54e60a053,69.857143,3.0,0.204499,4.035821,21,188.0,0


Duplicate rows in feature frame: 0


In [27]:
# Train an honest model while keeping entire clients together.
from sklearn.model_selection import GroupShuffleSplit
from sklearn.tree import DecisionTreeClassifier

features = [
    "feature_avg_daily_impressions",
    "feature_clicks",
    "feature_ctr",
    "feature_avg_position",
    "feature_active_days"
]

target = "future_decline_label"

# Remove incomplete rows, then give the remaining rows a clean index.
model_data = feature_frame.dropna(
    subset=features + [target]
).reset_index(drop=True)

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42
)

train_rows, test_rows = next(
    splitter.split(
        model_data,
        groups=model_data["client_hash_id"]
    )
)

def precision_at_50(actual, probabilities):
    results = pd.DataFrame({
        "actual": actual,
        "probability": probabilities
    })

    top_50 = results.nlargest(50, "probability")
    return top_50["actual"].mean()

honest_model = DecisionTreeClassifier(
    max_depth=4,
    random_state=42
)

honest_model.fit(
    model_data.loc[train_rows, features],
    model_data.loc[train_rows, target]
)

honest_probabilities = honest_model.predict_proba(
    model_data.loc[test_rows, features]
)[:, 1]

honest_score = precision_at_50(
    model_data.loc[test_rows, target].to_numpy(),
    honest_probabilities
)

print("Honest Precision@50:", round(honest_score, 3))

Honest Precision@50: 0.2


In [28]:
# Add one bad feature on purpose. It uses future information.
model_data["leaky_future_ratio"] = (
    model_data["future_avg_daily_impressions"] /
    model_data["feature_avg_daily_impressions"]
)

leaky_features = features + ["leaky_future_ratio"]

leaky_model = DecisionTreeClassifier(
    max_depth=4,
    random_state=42
)

leaky_model.fit(
    model_data.loc[train_rows, leaky_features],
    model_data.loc[train_rows, target]
)

leaky_probabilities = leaky_model.predict_proba(
    model_data.loc[test_rows, leaky_features]
)[:, 1]

leaky_score = precision_at_50(
    model_data.loc[test_rows, target].to_numpy(),
    leaky_probabilities
)

print("Honest Precision@50:", round(honest_score, 3))
print("Leaky Precision@50:", round(leaky_score, 3))

Honest Precision@50: 0.2
Leaky Precision@50: 1.0


In [29]:
# Remove the bad feature and keep the honest result.
model_data.drop(
    columns=["leaky_future_ratio"],
    inplace=True
)

print(
    "Leak removed:",
    "leaky_future_ratio" not in model_data.columns
)

print("Final score kept:", round(honest_score, 3))

Leak removed: True
Final score kept: 0.2


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

One limitation is that this slice keeps only pages with complete GSC availability across March and at least 100 earlier impressions. It may therefore exclude newer or lightly tracked pages. Results from this single month may also not represent seasonal patterns. The output is decision support and does not prove that refreshing a page will cause recovery.

In [30]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Final feature count:", len(features))
print("Final feature-frame rows:", len(feature_frame))
print("The deliberately leaked feature was removed.")

Final feature count: 5
Final feature-frame rows: 60796
The deliberately leaked feature was removed.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.